# Prepare a request of imagery at sites from a vendor using the CSDA Evaluation Sites GeoJSON

Paul Montesano, PhD  
June 2026

In [7]:
import pandas as pd
import geopandas as gpd
from datetime import datetime

### Read the CSDA Sites GeoJSON stored on GitHub

+ This GeoJSON is built directly off the CSDA Evaluation Sites Database.  
+ The notebook to process this GeoJSON is here: https://github.com/pahbs/csda_summaries/notebooks/csda_eval_sites_process.ipynb

In [27]:
RAW_BASE = 'https://raw.githubusercontent.com/pahbs/csda_summaries/master'
sites_url = f'{RAW_BASE}/sites/csda_sites_aoi.geojson'
sites = gpd.read_file(sites_url)

### Indicate a name for the vendor

In [29]:
VENDOR_NAME = 'Tanager'

In [30]:
# Get today's date
DATE = datetime.now().strftime('%Y%m%d')
DATE

'20260609'

In [31]:
sites.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 35 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Site Name abbrev            122 non-null    object  
 1   Site Name                   206 non-null    object  
 2   Location Name               143 non-null    object  
 3   Country                     206 non-null    object  
 4   Program Use                 143 non-null    object  
 5   Longitude                   143 non-null    float64 
 6   Latitude                    143 non-null    float64 
 7   Remote Sensing Domain       206 non-null    object  
 8   Priority Level              206 non-null    object  
 9   Evaluation Category         206 non-null    object  
 10  Source                      206 non-null    object  
 11  Surface Domain              206 non-null    object  
 12  Assessment type(s)          206 non-null    object  
 13  Resolution C

### Check some useful site attributes

In [32]:
print(list(sites['Evaluation Category'].unique()))

['Geometric', 'Radiometric', 'Radiometric & Geometric', 'InSAR']


### Each site's 'Site Name' is the key identifier for indicating the location of a CSDA request of data from a vendor

In [33]:
print(list(sites['Site Name'].unique()))

['Albuquerque', 'Amazon', 'Baotou', 'Atacama Desert', 'Belo Horizonte', 'Boston', 'Cairo', 'Cape Town', 'Casablanca', 'Caspian Sea', 'Catania', 'Crater Lake', 'Cuprite', 'Antarctica GPS', 'Doldrums', 'Atlantic Doldrums', 'Dublin', 'Gobabeb', 'Petermann Glacier', 'NISAR CR Array', 'Hohhot', 'Tianjin Docks', 'San Mateo Bridge', 'King Fahd Causeway', 'La Crau', 'Lake Pontchartrain Causeway', 'London', 'Melbourne', 'Navarre Causeway', 'DLR CR Array', 'Arabian Peninsula', 'Old Bahia Bridge', 'Phoenix', 'PICS Algeria-3', 'PICS Libya-1', 'PICS Libya-4', 'Piedmont', 'Railroad Valley', 'Buenos Aires', 'Rio Gallegos', 'RCRA', 'Salon-de-Provence', 'Sapporo', 'Suramadu Bridge', 'Shadnagar', 'Singapore', 'Sioux Falls', 'FMI CR Array', 'Valencia', 'Golmud', 'OPERA CR Array', 'WLEF', 'Etang de Berre', 'La Crau TIR', 'Lageren', 'Lake Constance', 'Lake Kasumigaura', 'Lake Tahoe', 'Myall Vale A', 'Oklahoma Agriculture Station', 'Pinnacles', 'Santarem', 'Salton Sea', 'Russell Ranch', 'Venice', 'AZ Coconi

In [41]:
def update_sites_attributes(sites_gdf, site_configs):
    """
    Update sites GeoDataFrame with attributes based on configuration.
    
    Parameters:
    -----------
    sites_gdf : GeoDataFrame
        Sites geodataframe to update
    site_configs : list of dict
        List of configurations, each with 'sites' and 'attributes' keys
        
    Returns:
    --------
    GeoDataFrame : Updated sites (copy)
    list : All site names from configs
    """
    sites_updated = sites_gdf.copy()
    all_sites = []
    
    for config in site_configs:
        site_list = config['sites']
        attributes = config['attributes']
        
        # Update attributes for these sites
        mask = sites_updated['Site Name'].isin(site_list)
        for key, value in attributes.items():
            sites_updated.loc[mask, key] = value
        
        all_sites.extend(site_list)
    
    return sites_updated, all_sites

### Indicate Time Series Sites for this request

*ENTER MANUALLY: this list should be the result of whatever randomization you apply to our full CSDA Evaluation Sites database*

In [ ]:
# TIMESERIES_SITES_FOR_GEOMETRIC_REQUEST = ['Albuquerque','Casablanca']
# TIMESERIES_SITES_FOR_RADIOMETRIC_REQUEST = []

In [ ]:
# # Types of request for each site
# # ...and the parameters needed to update the attributes in the sites database for this or
# TIMESERIES_SITE_DICT_GEOMETRIC = {
#     'ideal_num_acqs' = 10
# }
# # Any other types of sites?

In [ ]:
# OTHER_SITES_FOR_GEOMETRIC_REQUEST = ['Baotou']
# OTHER_SITES_FOR_RADIOMETRIC_REQUEST = ['WLEF','PICS Libya-4']

### Set a final list of CSDA sites for this request

In [36]:
# SITES_FOR_REQUEST = TIMESERIES_SITES_FOR_GEOMETRIC_REQUEST + \
#                     TIMESERIES_SITES_FOR_RADIOMETRIC_REQUEST + \
#                     OTHER_SITES_FOR_GEOMETRIC_REQUEST +\
#                     OTHER_SITES_FOR_RADIOMETRIC_REQUEST

### Update config of request parameters for sites chosen for this vendor request

In [43]:
# Usage
SITE_CONFIGS = [
    {
        'sites': ['Albuquerque', 'Casablanca'],
        'attributes': {
            'ideal_num_acqs': 10,
            'request_type': 'timeseries',
            'assessment_domain': 'geometric'
        }
    },
    {
        'sites': ['Baotou'],
        'attributes': {
            'ideal_num_acqs': 5,
            'request_type': 'other',
            'assessment_domain': 'geometric'
        }
    },
    {
        'sites': ['WLEF', 'PICS Libya-4'],
        'attributes': {
            'ideal_num_acqs': 3,
            'request_type': 'other',
            'assessment_domain': 'radiometric'
        }
    }
]

sites_updated, SITES_FOR_REQUEST = update_sites_attributes(sites, SITE_CONFIGS)
sites_subset = sites_updated[sites_updated['Site Name'].isin(SITES_FOR_REQUEST)]
sites_subset

,Site Name abbrev,Site Name,Location Name,Country,Program Use,Longitude,Latitude,Remote Sensing Domain,Priority Level,Evaluation Category,...,CR ID,Latitude (°),Longitude (°),Height Above Ellipsoid (m),Orientation (°),Elevation angle (°),Size (m),geometry,request_type,assessment_domain
0,Albuquerque,Albuquerque,New Mexico,USA,CSDA,-106.613826,35.068706,Optical Multi/Hyper,high,Geometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-106.59712 35.05540, -106.59764 35.0...",timeseries,geometric
2,Baotou,Baotou,China cal/val,China,CSDA,109.629437,40.851787,Optical Multi/Hyper,high,Radiometric & Geometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((109.62968 40.85161, 109.62967 40.851...",other,geometric
10,Casablanca,Casablanca,Casablanca,Morocco,CSDA,-7.622420,33.580370,Optical Multi/Hyper,high,Geometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-7.60648 33.56666, -7.60604 33.59371...",timeseries,geometric
37,PICS Libya-4,PICS Libya-4,PICS Libya-4,Libya,CSDA,23.390000,28.550000,Optical Multi/Hyper,high,Radiometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((23.39665 28.55833, 23.39110 28.53333...",other,radiometric
53,WLEF,WLEF,WLEF tower,USA,CSDA,-90.273200,45.944900,Optical Multi/Hyper,high,Radiometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-90.23454 45.94397, -90.23486 45.941...",other,radiometric


### Create a subset GeoJSON for this request

In [40]:
#sites_subset = sites[sites['Site Name'].isin(SITES_LIST_FOR_REQUEST)]

In [ ]:
OUTPUT_DIR = '/my/output/dir' # Specify your output dir here

In [ ]:
sites_subset.to_file(f'{OUTPUT_DIR/csda_sites_aoi_{VENDOR_NAME}_{DATE}.geojson}')